# Out-of-core example of single-cell analysis on the GPU

This notebook walks through what this capsule's `run` does, one step at a time, so you can look at
each result.

The counts are never loaded into memory all at once. Dask reads them in blocks of cells, and
rapids-singlecell processes each block on the GPU. That's how a million cells fit on one 15 GB T4.

```
open store ─► normalize ─► highly variable genes ─► scale ─► PCA ─► neighbors ─► UMAP ─► leiden
```

Based on rapids-singlecell's [out-of-core tutorial](https://rapids-singlecell.scverse.org/en/latest/notebooks/06_out-of-core.html)
by Severin Dicks (scverse), adapted to DataScales stores.

## 1. Start a GPU cluster

One Dask worker per GPU. With managed memory, a GPU that runs out of memory borrows host RAM
instead of failing.

In [ ]:
%matplotlib inline
import time

import anndata as ad
import cupy as cp
import dask
import numpy as np
import pandas as pd
import rapids_singlecell as rsc
import rmm
import scanpy as sc
import zarr
from anndata.experimental import read_elem_lazy
from annizarr import Repo
from dask.distributed import Client
from dask_cuda import LocalCUDACluster
from rmm.allocators.cupy import rmm_cupy_allocator

dask.config.set({"distributed.scheduler.worker-ttl": None})
cluster = LocalCUDACluster(rmm_managed_memory=True, enable_cudf_spill=True, rmm_allocator_external_lib_list="cupy")
client = Client(cluster)
rmm.reinitialize(managed_memory=True)
cp.cuda.set_allocator(rmm_cupy_allocator)
client

## 2. Open the store

Set `STORE` to your data asset. It can be a plain AnnData `.zarr`, a SpatialData `.zarr` (then
`TABLE` picks its table), or an icechunk repo holding either (then `BRANCH` picks the branch). It needs
raw counts in a sparse (CSR) `X`.


In [ ]:
STORE = "/data/<data asset>/<store>"
BRANCH = "qc"
TABLE = "table"
OBS_COLUMNS = ["sample_id"]
ROWS_PER_BLOCK = 24_000

root = Repo(STORE, branch=BRANCH).open_zarr("r") if Repo.exists(STORE) else zarr.open_group(STORE, mode="r")
if "X" not in root:
    root = root[f"tables/{TABLE}"]  # a SpatialData store: use its table
n_cells, n_genes = root["X"].attrs["shape"]
print(f"{n_cells:,} cells x {n_genes:,} genes")


## 3. Load the counts lazily

`X` becomes a Dask array: nothing is read yet. Of `obs`, only the columns in `OBS_COLUMNS` are
read, since reading every column of a large store takes a lot of memory.

In [ ]:
obs = root["obs"]
adata = ad.AnnData(
    X=read_elem_lazy(root["X"], (min(ROWS_PER_BLOCK, n_cells), n_genes)).astype(np.float32),
    obs=pd.DataFrame({c: ad.io.read_elem(obs[c]) for c in OBS_COLUMNS},
                     index=ad.io.read_elem(obs[obs.attrs["_index"]])),
    var=ad.io.read_elem(root["var"]),
)
rsc.get.anndata_to_GPU(adata)
adata

## 4. Normalize

Scale each cell to 10,000 counts, then take the log. Still lazy: this only adds steps to the plan.

In [ ]:
start = time.time()
rsc.pp.normalize_total(adata, target_sum=1e4)
rsc.pp.log1p(adata)

## 5. Find highly variable genes

This is the first step that reads the data. Keeping only the 2,000 most variable genes makes the
matrix small enough to hold on the GPU (`persist`).

In [ ]:
%%time
rsc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
adata = adata[:, adata.var["highly_variable"]].copy()
adata.X = adata.X.persist()

## 6. Scale and run PCA

Scaling divides each gene by its spread, capped at 10. PCA then reduces each cell to 50 numbers.
`.compute()` brings the result out of Dask.

In [ ]:
%%time
rsc.pp.scale(adata, max_value=10, zero_center=False)
rsc.pp.pca(adata, n_comps=50, svd_solver="covariance_eigh")
adata.obsm["X_pca"] = adata.obsm["X_pca"].compute().get()

## 7. Neighbors, UMAP and clusters

From here on the data is small, so one GPU works on it directly, without Dask.

In [ ]:
%%time
rsc.pp.neighbors(adata, n_neighbors=20, n_pcs=30, algorithm="ivfflat")
rsc.tl.umap(adata, min_dist=0.45, init_pos="spectral")
rsc.tl.leiden(adata, resolution=1.1)
print(f"{adata.obs['leiden'].nunique()} clusters in {time.time() - start:.0f}s")

## 8. Look at the result

In [ ]:
sc.pl.umap(adata, color=["leiden", *OBS_COLUMNS], ncols=2)

## 9. Shut down

Closing the cluster frees the GPU. This notebook writes nothing back; to save results to the
store, use `run` (see the README).

In [ ]:
client.close()
cluster.close()